In [ ]:
# Installiert die benötigte Gradio-Version für dieses Notebook
%pip install gradio==6.8.0

In [ ]:
import gradio as gr
import os
import subprocess
import sys
import pathlib

# --- LOGIK (UNVERÄNDERT AUS DER FUNKTIONIERENDEN VERSION) ---

def open_folder_dialog():
    """Öffnet den Dialog in einem Subprozess, um Freezing zu verhindern."""
    script = (
        "import tkinter as tk\n"
        "from tkinter import filedialog\n"
        "root = tk.Tk()\n"
        "root.attributes('-topmost', True)\n"
        "root.withdraw()\n"
        "folder = filedialog.askdirectory(title='Ordner auswählen')\n"
        "print(folder)\n"
    )
    try:
        result = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
        return result.stdout.strip()
    except Exception:
        return ""

def waehle_ordner(aktuell):
    pfad = open_folder_dialog()
    return pfad if pfad else aktuell

def berechne_pfade(ordner1, ordner2):
    if not ordner1 or not ordner2:
        return (
            ordner1 or '(kein Ordner gewählt)',
            ordner2 or '(kein Ordner gewählt)',
            '(beide Ordner auswählen)'
        )
    
    # Pfade normalisieren
    abs1 = str(pathlib.Path(ordner1).resolve())
    abs2 = str(pathlib.Path(ordner2).resolve())
    
    try:
        rel = os.path.relpath(abs2, abs1)
    except ValueError:
        rel = '(kein relativer Pfad möglich -- unterschiedliche Laufwerke)'
    
    return abs1, abs2, rel

# --- LAYOUT & CSS (AUS DEINEM ENTWURF) ---

css = '''
.path-display textarea {
    font-family: "JetBrains Mono", "Fira Mono", monospace !important;
    font-size: 0.88rem !important;
    background: #f8fafc !important;
    color: #334155 !important;
}
.result-box textarea {
    font-family: "JetBrains Mono", "Fira Mono", monospace !important;
    font-size: 0.9rem !important;
    background: #f0fdfa !important;
    color: #0f766e !important;
}
.rel-box textarea {
    font-family: "JetBrains Mono", "Fira Mono", monospace !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
    background: #fffbeb !important;
    color: #92400e !important;
}
'''

# --- GRADIO INTERFACE ---

with gr.Blocks(
    title='Pfad-Rechner',
    theme=gr.themes.Default(
        primary_hue='teal',
        neutral_hue='slate',
        font=[gr.themes.GoogleFont('Inter'), 'sans-serif']
    ),
    css=css
) as demo:

    gr.Markdown('# Relativer Pfad-Rechner')
    gr.Markdown(
        'Klicke auf **Ordner 1 wählen** bzw. **Ordner 2 wählen**.\n'
        'Es öffnet sich der native Datei-Dialog deines Systems.\n'
        'Dann **Pfade berechnen** drücken.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            btn1 = gr.Button('Ordner 1 wählen ...', variant='secondary')
        with gr.Column(scale=3):
            pfad1_anzeige = gr.Textbox(
                label='Ordner 1 -- Startpunkt',
                placeholder='(noch kein Ordner gewählt)',
                interactive=False,
                elem_classes=['path-display']
            )

    with gr.Row():
        with gr.Column(scale=1):
            btn2 = gr.Button('Ordner 2 wählen ...', variant='secondary')
        with gr.Column(scale=3):
            pfad2_anzeige = gr.Textbox(
                label='Ordner 2 -- Zielpunkt',
                placeholder='(noch kein Ordner gewählt)',
                interactive=False,
                elem_classes=['path-display']
            )

    # Event-Handler für die Ordnerwahl
    btn1.click(fn=waehle_ordner, inputs=[pfad1_anzeige], outputs=[pfad1_anzeige])
    btn2.click(fn=waehle_ordner, inputs=[pfad2_anzeige], outputs=[pfad2_anzeige])

    berechnen_btn = gr.Button('Pfade berechnen', variant='primary', size='lg')
 
    with gr.Row():
        abs_pfad1 = gr.Textbox(
            label='a) Absoluter Pfad -- Ordner 1',
            interactive=False,
            buttons=['copy'],
            elem_classes=['result-box']
        )
        abs_pfad2 = gr.Textbox(
            label='b) Absoluter Pfad -- Ordner 2',
            interactive=False,
            buttons=['copy'],
            elem_classes=['result-box']
        )
 
    rel_pfad = gr.Textbox(
        label='c) Relativer Pfad (von Ordner 1 nach Ordner 2)',
        interactive=False,
        buttons=['copy'],
        elem_classes=['rel-box']
    )

    # Berechnung auslösen
    berechnen_btn.click(
        fn=berechne_pfade,
        inputs=[pfad1_anzeige, pfad2_anzeige],
        outputs=[abs_pfad1, abs_pfad2, rel_pfad]
    )

    gr.Markdown(
        '---\n'
        'Legende: `..` = eine Ebene nach oben | '
        '`../..` = zwei Ebenen hoch | '
        '`subfolder/name` = direkt unterhalb von Ordner 1'
    )

# Start der App
demo.launch(inbrowser=True)